# Volume 3, Chapter 10: Martingales (Python companion)

Exercises first, solutions at the end. We compute the waiting time of any word with the border array of the Knuth–Morris–Pratt algorithm, play optional stopping against a casino with a credit limit, watch Pólya's urn and branching populations converge to random limits, test Azuma's inequality on balls in bins and on the longest increasing subsequence, and use Doob's maximal inequality to make an A/B test that may be checked continuously without inflating its false alarm rate.

*Before each solution, predict what the output will look like.*

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(10)

## Exercises

**P10.1 (★) Words, monkeys and KMP.** The overlaps of a word $w$ (Theorem 10.14) are its *borders*: the lengths $k$ such that the prefix of length $k$ is also a suffix. The Knuth–Morris–Pratt string-matching algorithm computes, for each prefix of $w$, its longest proper border. Use it to list all borders of $w$ and compute $\mathbb E[T]$ for ABRACADABRA, for the DNA words ATAT and ATCG, and for HTH and HHT. Check the coin and DNA values by simulation.

**P10.2 (★) Stopping when ahead.** Simulate the strategy ``play a fair coin, stop as soon as $1$ euro ahead'' with a credit limit: the player also stops when her loss reaches $L$. For $L=10,100,1000$, estimate the probability of leaving with $+1$, the mean gain, and the mean duration, and compare with Proposition 10.13.

**P10.3 (★) Pólya's urn.** Simulate $20\,000$ urns for $1000$ draws, starting from one red and one blue ball, and compare the histogram of the final proportion with the uniform law (Theorem 10.21). Repeat with $2$ red and $1$ blue and compare with the Beta$(2,1)$ density $2x$ (Exercise 10.5).

**P10.4 (★★) Branching growth.** For Poisson$(1.5)$ offspring, simulate $5000$ populations for $15$ generations. Plot a few paths of $W_n=Z_n/1.5^n$, and compare the fraction of runs with $W_{15}=0$ with the extinction probability $q$ (solve $q=e^{1.5(q-1)}$), and the mean and variance of $W_{15}$ with Theorem 10.22.

**P10.5 (★★) Azuma in practice.** (a) Throw $m=n=10^4$ balls into bins, $5000$ times, and compare the spread of the number of empty bins with the bound of Theorem 10.25. (b) The length of the longest increasing subsequence of $n$ random numbers changes by at most $1$ when one number changes. Compute it with patience sorting for $n=10^4$, and compare its spread with the bound. What does Azuma get right, and what does it miss?

**P10.6 (★) The random harmonic series.** Plot a few paths of $\sum_{k\le n}\frac{\varepsilon_k}k$ and the histogram of its limit over many runs; check that the variance is $\frac{\pi^2}6$ (Exercise 10.6).

**P10.7 (★★) A/B testing with continuous monitoring.** Visitors convert with probability $p_0=0.10$ under the old design. A team tests whether a new design converts at $p_1=0.12$, and computes after each visitor the likelihood ratio $L_n=\prod_{k\le n}\frac{q(X_k)}{p(X_k)}$ of Exercise 10.8, with $p$ = Bernoulli$(0.10)$ and $q$ = Bernoulli$(0.12)$. It declares the new design better as soon as $L_n\ge20$. When the truth is $p_0$ (no improvement), show by Theorem 10.15 that the probability of a false alarm, however long the test runs, is at most $\frac1{20}$, and check it by simulation. Compare with the naive approach that runs a $5\%$ test of the proportion after every $100$ visitors.

## Solutions

**P10.1.** The KMP prefix function $\pi[i]$ is the length of the longest proper border of $w_1\cdots w_{i+1}$. The borders of $w$ are $\pi[\ell-1]$, then the borders of that border, and so on, plus $\ell$ itself.

In [ ]:
def prefix_function(w):
    pi = [0] * len(w)
    k = 0
    for i in range(1, len(w)):
        while k and w[i] != w[k]:
            k = pi[k - 1]
        if w[i] == w[k]:
            k += 1
        pi[i] = k
    return pi
def borders(w):
    pi, b, k = prefix_function(w), [len(w)], None
    k = pi[-1]
    while k:
        b.append(k); k = pi[k - 1]
    return b
def expected_wait(w, m):
    return sum(m**k for k in borders(w))
for w, m in [('ABRACADABRA', 26), ('ABRACADABRX', 26), ('ATAT', 4), ('ATCG', 4), ('HTH', 2), ('HHT', 2), ('HH', 2), ('HT', 2)]:
    print(f'{w:12s} borders {sorted(borders(w))!s:14s} E[T] = {expected_wait(w, m):,}')

In [ ]:
def simulate_wait(w, alphabet, reps):
    L = len(w); total = 0
    idx = {c: i for i, c in enumerate(alphabet)}
    target = [idx[c] for c in w]
    for _ in range(reps):
        buf, t = [], 0
        while True:
            chunk = rng.integers(len(alphabet), size=4096)
            for c in chunk:
                t += 1; buf.append(int(c))
                if len(buf) > L: buf.pop(0)
                if buf == target: break
            else:
                continue
            break
        total += t
    return total / reps
for w, alph in [('HTH', 'HT'), ('HHT', 'HT'), ('ATAT', 'ACGT'), ('ATCG', 'ACGT')]:
    print(w, 'simulated', simulate_wait(w, alph, 20000), '  theory', expected_wait(w, len(alph)))

The monkey's theorem reduces to the border array, which KMP computes in linear time; KMP itself uses the borders for the opposite purpose, to know how far to shift after a mismatch. ATAT and ATCG appear equally often in a long random DNA sequence, but ATAT is first seen later on average ($272$ letters against $256$), because its occurrences overlap. In genomics, the same computation gives the expected distance to the first occurrence of a motif, and statistical tests for over-represented words must take overlaps into account.

**P10.2.**

In [ ]:
def stop_when_ahead(L, reps):
    gains, durations = np.empty(reps), np.empty(reps)
    for r in range(reps):
        s, t = 0, 0
        while -L < s < 1:
            # play in vectorised chunks of steps until the walk exits
            steps = rng.choice([-1, 1], size=256)
            path = s + np.cumsum(steps)
            out = np.flatnonzero((path <= -L) | (path >= 1))
            if len(out):
                t += out[0] + 1; s = path[out[0]]
            else:
                t += 256; s = path[-1]
        gains[r], durations[r] = s, t
    return gains, durations
for L, reps in [(10, 20000), (100, 20000), (1000, 2000)]:
    g, d = stop_when_ahead(L, reps)
    print(f'L={L:5d}: P(+1) = {np.mean(g == 1):.4f} (theory {L / (L + 1):.4f})   mean gain {g.mean():+.3f}   mean duration {d.mean():8.1f} (theory {L})')

The player leaves with a profit with probability $\frac L{L+1}$, but the rare ruins cost $L$ each, and the mean gain is $0$ up to sampling error: Proposition 10.13 with $a=L$ and $N=L+1$, which also gives the mean duration $a(N-a)=L$. As $L\to\infty$ the probability of profit tends to $1$, the mean duration to infinity, and the rare ruin becomes a catastrophe. With $L=1000$ the sample means are noisy, because a few ruins of $-1000$, after very long games, dominate them: exactly the kind of risk that a strategy with a high ``win rate'' hides.

**P10.3.**

In [ ]:
def polya(a, b, n, reps):
    red = np.full(reps, a, dtype=float); tot = a + b
    for _ in range(n):
        red += rng.random(reps) < red / tot
        tot += 1
    return red / tot
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
x = np.linspace(0, 1, 200)
for k, (a, b, dens) in enumerate([(1, 1, np.ones_like(x)), (2, 1, 2 * x)]):
    M = polya(a, b, 1000, 20000)
    ax[k].hist(M, bins=40, density=True, alpha=0.6)
    ax[k].plot(x, dens, 'k')
    ax[k].set_title(f'start: {a} red, {b} blue;  mean {M.mean():.3f}')
plt.tight_layout(); plt.show()

The final proportions are spread uniformly for the $(1,1)$ start, and with density $2x$ for the $(2,1)$ start, of mean $\frac23$, the initial proportion, as the martingale property demands. Reinforcement is a simple model of the ``rich get richer'' effect in networks: Chapter 12 and the preferential attachment graph of the Chapter 8 notebook are relatives.

**P10.4.**

In [ ]:
m, gens, reps = 1.5, 15, 5000
Z = np.ones(reps, dtype=np.int64)
W = np.empty((gens + 1, reps)); W[0] = 1
for n in range(1, gens + 1):
    Z = rng.poisson(m * Z)          # a sum of Z independent Poisson(m) is Poisson(m Z)
    W[n] = Z / m**n
q = 0.0
for _ in range(200):
    q = math.exp(m * (q - 1))
print('extinction probability q =', round(q, 4), '  fraction with W_15 = 0:', np.mean(W[-1] == 0))
print('mean of W_15:', W[-1].mean(), '  (theory 1)')
print('variance of W_15:', W[-1].var(), '  theory for W:', m / (m * (m - 1)), ' (sigma^2 = m for Poisson)')
plt.figure(figsize=(7, 3))
for r in range(12):
    plt.plot(W[:, r], marker='.', lw=0.8)
plt.xlabel('generation n'); plt.ylabel('$Z_n / m^n$'); plt.tight_layout(); plt.show()
plt.figure(figsize=(7, 3)); plt.hist(W[-1][W[-1] > 0], bins=60, density=True); plt.title('law of W on survival'); plt.tight_layout(); plt.show()

The paths settle after a few generations: a population that has survived the first few generations grows like $Wm^n$, with $W$ fixed early. About $41.7\%$ of the runs die out, matching $q\approx0.417$, and these are exactly the runs with $W=0$ (Theorem 10.22). The mean and variance of $W_{15}$ are close to $1$ and $\frac{\sigma^2}{m(m-1)}=2$ (the variance of $W_{15}$ is $\sigma^2\sum_{k<15}m^{-k-2}$, within $0.3\%$ of the limit). Using the identity ``a sum of $Z$ Poisson$(m)$ variables is Poisson$(mZ)$'' makes the simulation of $5000$ populations of up to $10^5$ individuals instantaneous.

**P10.5.** Patience sorting: deal the numbers one by one onto piles, each on the leftmost pile whose top is larger (or a new pile); the number of piles is the length of the longest increasing subsequence, and binary search makes it $O(n\log n)$.

In [ ]:
import bisect
def lis(seq):
    tops = []
    for v in seq:
        i = bisect.bisect_left(tops, v)
        if i == len(tops): tops.append(v)
        else: tops[i] = v
    return len(tops)
n = m = 10_000; reps = 5000
empty = np.array([n - len(np.unique(rng.integers(n, size=m))) for _ in range(reps)])
print('empty bins: mean', empty.mean(), ' (n(1-1/n)^m =', n * (1 - 1 / n)**m, ')  std', empty.std())
t = np.quantile(np.abs(empty - empty.mean()), 0.99)
print('  99% of deviations below', t, '  Azuma: P(|dev| >= t) <= 2exp(-t^2/2m) gives t =', math.sqrt(2 * m * math.log(200)))
L = np.array([lis(rng.random(n)) for _ in range(1000)])
print('LIS: mean', L.mean(), ' (2 sqrt n =', 2 * math.sqrt(n), ')  std', L.std())
print('  Azuma 99% window:', math.sqrt(2 * n * math.log(200)), '  observed 99% deviation:', np.quantile(np.abs(L - L.mean()), 0.99))

For the empty bins, Azuma's $99\%$ window is about $326$ against an observed $\approx80$: the right $\sqrt m$ scale, with a constant about $4$ times too large (McDiarmid's factor $2$ in the exponent would halve it). For the longest increasing subsequence the bound is the same $\sqrt n$ window, but the true fluctuations are of order $n^{1/6}\approx4.6$: Baik, Deift and Johansson proved in 1999 that they follow the Tracy–Widom law of random matrix theory. Azuma only uses the bounded differences, so it cannot see that most changes of one number do not change the LIS at all. The mean $\approx193$ is below $2\sqrt n=200$ by a correction of order $n^{1/6}$, also predicted by that theory.

**P10.6.**

In [ ]:
N, reps = 10_000, 4000
k = np.arange(1, N + 1)
plt.figure(figsize=(7, 3))
for _ in range(6):
    plt.plot(k, np.cumsum(rng.choice([-1, 1], size=N) / k), lw=0.8)
plt.xscale('log'); plt.xlabel('n'); plt.ylabel('partial sum'); plt.tight_layout(); plt.show()
limits = np.array([np.sum(rng.choice([-1, 1], size=N) / k) for _ in range(reps)])
print('variance of the sum up to 10^4:', limits.var(), '  pi^2/6 =', math.pi**2 / 6)
plt.figure(figsize=(7, 3)); plt.hist(limits, bins=60, density=True); plt.tight_layout(); plt.show()

The paths stabilise once $n$ reaches a few hundred, each at its own random limit. The limit law has variance $\frac{\pi^2}6\approx1.645$, but it is not Gaussian: the first few terms weigh too much for a central limit effect. Every real number is the sum for a suitable choice of signs (choose them greedily, since $\sum\frac1n$ diverges and the terms tend to $0$), yet large values are extremely rare: reaching $x$ needs roughly the first $e^x$ signs to agree, so the tails decay much faster than Gaussian ones (B.~Schmuland, *Random harmonic series*, American Mathematical Monthly, 2003).

**P10.7.** Under $p_0$, $L_n$ is a non-negative martingale with $L_0=1$, so by Theorem 10.15, $P(\max_{k\le n}L_k\ge20)\le\frac{\mathbb E[L_n]}{20}=\frac1{20}$ for every $n$, hence for the whole infinite test (this is *Ville's inequality*, 1939).

In [ ]:
p0, p1, horizon, reps = 0.10, 0.12, 20_000, 2000
up, down = math.log(p1 / p0), math.log((1 - p1) / (1 - p0))
false_lr, false_naive = 0, 0
for _ in range(reps):
    x = rng.random(horizon) < p0                      # truth: no improvement
    logL = np.cumsum(np.where(x, up, down))
    if logL.max() >= math.log(20): false_lr += 1
    # naive: z-test of the proportion against p0 after every 100 visitors
    n = np.arange(100, horizon + 1, 100)
    phat = np.cumsum(x)[n - 1] / n
    z = (phat - p0) / np.sqrt(p0 * (1 - p0) / n)
    if (z > 1.645).any(): false_naive += 1
print('false alarm rate, likelihood ratio >= 20:', false_lr / reps, '  (Doob: at most 0.05)')
print('false alarm rate, naive repeated 5% test:', false_naive / reps)
# power when the improvement is real
hits, times = 0, []
for _ in range(500):
    x = rng.random(horizon) < p1
    logL = np.cumsum(np.where(x, up, down))
    hit = np.flatnonzero(logL >= math.log(20))
    if len(hit): hits += 1; times.append(hit[0] + 1)
print('true improvement detected in', hits / 500, 'of the runs, median number of visitors', np.median(times))

The likelihood ratio test raises a false alarm in about $4.5\%$ of the runs, below Doob's $5\%$ (the bound is not quite reached because $L_n$ jumps over $20$ when it crosses it, and the test stops at $20\,000$ visitors), however often it is checked. The naive procedure, a correct $5\%$ test applied $200$ times, raises a false alarm in almost $40\%$ of the runs: ``peeking'' at a classical test inflates its error rate, a well-known pitfall of online experiments. When the improvement is real, the likelihood ratio detects it in every run, after about a thousand visitors (median). Tests built on non-negative martingales, called *e-values* or *anytime-valid* tests, are now used by large online platforms for exactly this reason.